# Notebook 2 — Build the Splits

This builds the two ways of splitting the data that your whole thesis compares:

| Split | How | What it represents |
|---|---|---|
| **Honest** | patient-level — a patient is entirely in train **or** test, never both | correct practice |
| **Leaky** | random slice-level — a patient's slices scatter across train and test | what many papers actually do |

The gap between these two is your headline result (RQ2).

 Run top to bottom. Cell 5 contains a hard leakage test — if it fails, stop.

In [1]:
# Cell 1 — imports
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
print("imports ok")

imports ok


## Cell 2 — paths and settings

`N_FOLDS = 5` matches common practice on this dataset. `SEED` fixes reproducibility.

In [2]:
# Cell 2 — the only cell you check
IN_DIR  = "/kaggle/input"          # where your Notebook-1 output is mounted (read-only)
OUT_DIR = "/kaggle/working/data"
N_FOLDS  = 5
SEED     = 42

In [3]:
# Cell 3 — find and load the manifest (auto-search under /kaggle/input)
from pathlib import Path
hits = list(Path(IN_DIR).rglob("manifest.csv"))
assert hits, f"No manifest.csv found under {IN_DIR}. Did you add Notebook 1's output as a dataset?"
manifest_path = hits[0]
print("found:", manifest_path)

manifest = pd.read_csv(manifest_path)
print("loaded", len(manifest), "slices,", manifest.pid.nunique(), "patients")
manifest.head()

found: /kaggle/input/notebooks/souravpaul100304/1-cheng-dataset-analysis/data/manifest.csv
loaded 3064 slices, 233 patients


,file_id,pid,label,label_name,orig_size,mask_pixels
0,1,100360,1,meningioma,512x512,873
1,2,100360,1,meningioma,512x512,1656
2,3,100360,1,meningioma,512x512,2018
3,4,100360,1,meningioma,512x512,1699
4,5,100360,1,meningioma,512x512,2148


## Cell 4 — build both splits

`StratifiedGroupKFold` with `groups=pid` keeps every patient inside one fold (honest) while balancing classes. Plain `StratifiedKFold` ignores patients, so siblings scatter (leaky). Each slice gets a `fold_honest` and a `fold_leaky` number (0–4).

In [4]:
# Cell 4 — assign folds
y      = manifest["label"].values
groups = manifest["pid"].values

manifest["fold_honest"] = -1
manifest["fold_leaky"]  = -1

# HONEST: groups=pid -> a whole patient stays in one fold
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, test_idx) in enumerate(sgkf.split(manifest, y, groups)):
    manifest.iloc[test_idx, manifest.columns.get_loc("fold_honest")] = fold

# LEAKY: no groups -> a patient's slices scatter across folds
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, test_idx) in enumerate(skf.split(manifest, y)):
    manifest.iloc[test_idx, manifest.columns.get_loc("fold_leaky")] = fold

print("folds assigned")
print("honest fold sizes:", manifest.fold_honest.value_counts().sort_index().tolist())
print("leaky  fold sizes:", manifest.fold_leaky.value_counts().sort_index().tolist())

folds assigned
honest fold sizes: [645, 536, 602, 685, 596]
leaky  fold sizes: [613, 613, 613, 613, 612]


## Cell 5 — the leakage test

For each fold: what fraction of test slices come from a patient who is **also** in that fold's training set?

- Honest **must** be 0%. The assertion enforces it — if it ever fails, the split is broken and nothing downstream is valid.
- Leaky will be very high. it's evidence about what random splitting actually does.

In [5]:
# Cell 5 — measure and enforce
def contamination(col):
    rates = []
    for fold in range(N_FOLDS):
        test  = manifest[manifest[col] == fold]
        train = manifest[manifest[col] != fold]
        shared = set(train.pid) & set(test.pid)
        rates.append(100 * test.pid.isin(shared).sum() / len(test))
    return np.array(rates)

honest = contamination("fold_honest")
leaky  = contamination("fold_leaky")

print("Contaminated test slices (same patient also in training):")
print(f"  HONEST : {honest.mean():5.1f}%   per fold {np.round(honest,1).tolist()}")
print(f"  LEAKY  : {leaky.mean():5.1f}%   per fold {np.round(leaky,1).tolist()}")

assert honest.max() == 0.0, "PATIENT LEAKAGE IN HONEST SPLIT — STOP, DO NOT CONTINUE"
print("\n[PASS] Honest split is patient-disjoint.")

Contaminated test slices (same patient also in training):
  HONEST :   0.0%   per fold [0.0, 0.0, 0.0, 0.0, 0.0]
  LEAKY  :  99.7%   per fold [99.3, 99.8, 100.0, 99.7, 99.7]

[PASS] Honest split is patient-disjoint.


## Cell 6 — check class balance per fold

Each fold should have roughly the same class proportions. Big swings mean stratification struggled (possible with only 233 patients) 

In [6]:
# Cell 6 — class balance
tab = pd.crosstab(manifest.fold_honest, manifest.label_name, normalize="index") * 100
print("Class balance per honest fold (%):")
print(tab.round(1).to_string())

Class balance per honest fold (%):
label_name   glioma  meningioma  pituitary
fold_honest                               
0              44.2        24.2       31.6
1              39.2        25.4       35.4
2              44.7        25.6       29.7
3              49.3        19.3       31.4
4              54.4        21.8       23.8


## Cell 7 — save

Writes `manifest_splits.csv` — the manifest plus the two fold columns.

In [7]:
# Cell 7 — save
out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
out_path = out / "manifest_splits.csv"
manifest.to_csv(out_path, index=False)
print("saved", out_path.resolve())
print("columns:", list(manifest.columns))

saved /kaggle/working/data/manifest_splits.csv
columns: ['file_id', 'pid', 'label', 'label_name', 'orig_size', 'mask_pixels', 'fold_honest', 'fold_leaky']
